# Blender Redesign — Regime-Conditioned Model Performance

**Goal:** Quantify which model strategies work best in each regime group,
then derive optimal weights for the RegimeEnsembleBlender.

**Key findings from regime_foundation:**
- TREND_BULL → negative forward returns (mean-reversion opportunity)
- TREND_BEAR → positive forward returns (mean-reversion opportunity)
- p_trending is bimodal → hard gates, not soft blending
- 9→4 grouping needs redesign (BULL/BEAR matters more than CLEAN/VOLATILE)
- MTF confirming moves are 12-33% larger (validated)

In [1]:
import sys, types
sys.path.insert(0, '../src')

# Create 'app' namespace alias (regime module uses app.regime.* imports)
app = types.ModuleType('app')
app.__path__ = ['../src/libs', '../src/apps']
sys.modules['app'] = app

import numpy as np
import pandas as pd
import time
from datetime import datetime, timezone
import warnings
warnings.filterwarnings('ignore')

from binance.um_futures import UMFutures
from libs.features.indicators.momentum.rsi import _compute_rsi_batch
from libs.features.indicators.momentum.macd import _compute_macd_batch
from libs.features.indicators.momentum.cci import _compute_cci_batch
from libs.features.indicators.momentum.mfi import _compute_mfi_batch
from libs.features.indicators.momentum.adx import _compute_adx_batch
from libs.features.indicators.volatility.atr import _compute_atr_batch
from libs.features.indicators.volatility.bollinger import _compute_bb_batch
from libs.features.indicators.volatility.keltner import _compute_keltner_batch
from libs.features.indicators.trend.kama import _compute_kama_batch
from libs.features.indicators.trend.ema import _compute_ema_batch
from libs.regime.orchestrator import RegimeOrchestrator

client = UMFutures()
print('Setup complete')

Setup complete


## 1. Fetch Data & Compute Regime + Indicators

In [2]:
# ── OHLCV fetch ────────────────────────────────────────────────────
_ALL_COLS = [
    'timestamp', 'open', 'high', 'low', 'close', 'volume', 'close_time',
    'quote_volume', 'trades', 'taker_buy_base', 'taker_buy_quote', 'ignore'
]

def fetch_ohlcv(symbol: str, interval: str, start_date: str, end_date: str) -> pd.DataFrame:
    start_ms = int(datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    end_ms = int(datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=timezone.utc).timestamp() * 1000)
    all_rows = []
    cursor = start_ms
    while cursor < end_ms:
        raw = client.klines(symbol, interval, startTime=cursor, endTime=end_ms, limit=1500)
        if not raw:
            break
        all_rows.extend(raw)
        cursor = int(raw[-1][6]) + 1
        if len(raw) < 1500:
            break
        time.sleep(0.15)
    df = pd.DataFrame(all_rows, columns=_ALL_COLS)
    for c in ['open','high','low','close','volume','taker_buy_base']:
        df[c] = df[c].astype(float)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df = df.drop_duplicates('timestamp').sort_values('timestamp').reset_index(drop=True)
    print(f'{symbol} {interval}: {len(df)} bars [{df.timestamp.iloc[0]} → {df.timestamp.iloc[-1]}]')
    return df

SYMBOLS = ['BTCUSDT', 'ETHUSDT']
START, END = '2025-06-01', '2026-05-31'

data = {}
for sym in SYMBOLS:
    data[sym] = fetch_ohlcv(sym, '1h', START, END)

print(f'\nTotal: {sum(len(v) for v in data.values())} bars')

BTCUSDT 1h: 8737 bars [2025-06-01 00:00:00+00:00 → 2026-05-31 00:00:00+00:00]
ETHUSDT 1h: 8737 bars [2025-06-01 00:00:00+00:00 → 2026-05-31 00:00:00+00:00]

Total: 17474 bars


In [5]:
# ── Compute regime labels + all indicators ──────────────────────────
results = {}

for sym in SYMBOLS:
    print(f'\n--- {sym} ---')
    df = data[sym].copy()
    o, h, l, c, v = df['open'].values, df['high'].values, df['low'].values, df['close'].values, df['volume'].values
    n = len(c)
    
    # Regime
    orch = RegimeOrchestrator.create(sym, '1h')
    regime_df = orch.analyze_series(df)
    df['regime'] = regime_df['regime'].values[:n]
    df['p_trending'] = regime_df['p_trending'].values[:n]
    df['changepoint_prob'] = regime_df['changepoint_prob'].values[:n]
    
    # Core indicators
    df['RSI'] = _compute_rsi_batch(c, 14)
    df['ATR'] = _compute_atr_batch(h, l, c, 14)
    adx_arr, plus_di, minus_di = _compute_adx_batch(h, l, c, 14)
    df['ADX'] = adx_arr
    df['CCI'] = _compute_cci_batch(h, l, c, 20)
    df['MFI'] = _compute_mfi_batch(h, l, c, v, 14)
    bb_mid, bb_upper, bb_lower = _compute_bb_batch(c, 20, 2.0)
    df['BB_upper'] = bb_upper
    df['BB_lower'] = bb_lower
    df['BB_mid'] = bb_mid
    kc_mid, kc_upper, kc_lower = _compute_keltner_batch(h, l, c, 20, 1.5, 10)
    df['KC_upper'] = kc_upper
    df['KC_lower'] = kc_lower
    df['KAMA'] = _compute_kama_batch(c, 10, 2, 30)
    df['EMA_fast'] = _compute_ema_batch(c, 12, 2.0 / (12 + 1))
    df['EMA_slow'] = _compute_ema_batch(c, 26, 2.0 / (26 + 1))
    macd_line, macd_signal, macd_hist = _compute_macd_batch(c, 12, 26, 9)
    df['MACD_hist'] = macd_hist
    
    # Forward returns
    for h_val in [1, 3, 6, 12, 24]:
        df[f'fwd_ret_{h_val}'] = df['close'].pct_change(h_val).shift(-h_val)
    
    # Squeeze detection
    df['squeeze'] = (df['BB_upper'] < df['KC_upper']) & (df['BB_lower'] > df['KC_lower'])
    
    results[sym] = df
    print(f'  Regimes: {df["regime"].value_counts().to_dict()}')
    print(f'  Indicators computed: RSI, ATR, ADX, CCI, MFI, BB, KC, KAMA, EMA, MACD')


--- BTCUSDT ---


  Regimes: {'QUIET_MR_SQUEEZE': 1883, 'CHOPPY': 1814, 'QUIET_MR_RANGE': 1661, 'CLEAN_TREND_FLAT': 944, 'CLEAN_TREND_BULL': 595, 'CLEAN_TREND_BEAR': 585, 'VOLATILE_TREND_BEAR': 464, 'VOLATILE_TREND_FLAT': 449, 'VOLATILE_TREND_BULL': 342}
  Indicators computed: RSI, ATR, ADX, CCI, MFI, BB, KC, KAMA, EMA, MACD

--- ETHUSDT ---


  Regimes: {'CHOPPY': 2128, 'QUIET_MR_SQUEEZE': 1921, 'QUIET_MR_RANGE': 1142, 'CLEAN_TREND_FLAT': 938, 'VOLATILE_TREND_FLAT': 633, 'CLEAN_TREND_BEAR': 597, 'VOLATILE_TREND_BEAR': 502, 'CLEAN_TREND_BULL': 461, 'VOLATILE_TREND_BULL': 415}
  Indicators computed: RSI, ATR, ADX, CCI, MFI, BB, KC, KAMA, EMA, MACD


## 2. Simulate Model Signals Per Regime

Replicate the 3 core model signal logic and measure performance per regime.

In [6]:
# ── Model signal generators ────────────────────────────────────────

def mr_signals(df: pd.DataFrame) -> pd.Series:
    """MeanReversion: ADX < 25, RSI < 30 → long, RSI > 70 → short."""
    adx_gate = df['ADX'] < 25
    rsi = df['RSI']
    bb_l, bb_u = df['BB_lower'], df['BB_upper']
    close = df['close']
    
    long_cond = adx_gate & (rsi < 30) & (close < bb_l)
    short_cond = adx_gate & (rsi > 70) & (close > bb_u)
    return pd.Series(np.where(long_cond, 1, np.where(short_cond, -1, 0)), index=df.index)

def momentum_signals(df: pd.DataFrame) -> pd.Series:
    """Momentum: RSI > 70 + MACD hist > 0 → long, RSI < 34 + hist < 0 → short."""
    long_cond = (df['RSI'] > 70) & (df['MACD_hist'] > 0)
    short_cond = (df['RSI'] < 34) & (df['MACD_hist'] < 0)
    return pd.Series(np.where(long_cond, 1, np.where(short_cond, -1, 0)), index=df.index)

def squeeze_signals(df: pd.DataFrame) -> pd.Series:
    """SqueezeBreakout: squeeze release + KAMA direction."""
    was_squeeze = df['squeeze'].shift(1, fill_value=False)
    released = was_squeeze & ~df['squeeze']
    kama_up = df['KAMA'] > df['KAMA'].shift(1)
    long_cond = released & kama_up
    short_cond = released & ~kama_up
    return pd.Series(np.where(long_cond, 1, np.where(short_cond, -1, 0)), index=df.index)

# Generate signals for all
for sym in SYMBOLS:
    df = results[sym]
    df['sig_mr'] = mr_signals(df)
    df['sig_mom'] = momentum_signals(df)
    df['sig_sq'] = squeeze_signals(df)
    print(f'{sym}: MR={df["sig_mr"].abs().sum()}, Mom={df["sig_mom"].abs().sum()}, SQ={df["sig_sq"].abs().sum()} signals')

BTCUSDT: MR=107, Mom=1239, SQ=234 signals
ETHUSDT: MR=122, Mom=1363, SQ=252 signals


## 3. Per-Regime Model Performance Matrix

In [7]:
# ── Regime groupings to test ──────────────────────────────────────
# Current grouping (5 groups, no bull/bear split)
CURRENT_GROUPS = {
    'CLEAN_TREND_BULL': 'CLEAN_TREND', 'CLEAN_TREND_BEAR': 'CLEAN_TREND', 'CLEAN_TREND_FLAT': 'CLEAN_TREND',
    'VOLATILE_TREND_BULL': 'VOLATILE_TREND', 'VOLATILE_TREND_BEAR': 'VOLATILE_TREND', 'VOLATILE_TREND_FLAT': 'VOLATILE_TREND',
    'QUIET_MR_RANGE': 'QUIET_RANGE', 'QUIET_MR_SQUEEZE': 'SQUEEZE', 'CHOPPY': 'CHOPPY',
}

# Proposed grouping (4 groups, bull/bear split)
PROPOSED_GROUPS = {
    'CLEAN_TREND_BULL': 'TREND_BULL', 'VOLATILE_TREND_BULL': 'TREND_BULL',
    'CLEAN_TREND_BEAR': 'TREND_BEAR', 'VOLATILE_TREND_BEAR': 'TREND_BEAR',
    'CLEAN_TREND_FLAT': 'RANGE', 'QUIET_MR_RANGE': 'RANGE', 'QUIET_MR_SQUEEZE': 'RANGE',
    'VOLATILE_TREND_FLAT': 'CHOPPY', 'CHOPPY': 'CHOPPY',
}

MODELS = ['sig_mr', 'sig_mom', 'sig_sq']
MODEL_LABELS = {'sig_mr': 'MeanReversion', 'sig_mom': 'Momentum', 'sig_sq': 'SqueezeBreakout'}

def backtest_model_per_regime(df, sig_col, horizon=12, atr_tp=2.0, atr_sl=1.5):
    """Backtest a signal column with ATR-based TP/SL, return per-trade results."""
    trades = []
    entries = df[df[sig_col] != 0].copy()
    
    for idx in entries.index:
        if idx + horizon >= len(df):
            continue
        direction = int(entries.loc[idx, sig_col])
        entry_price = df.loc[idx, 'close']
        atr = df.loc[idx, 'ATR']
        regime = df.loc[idx, 'regime']
        
        if np.isnan(atr) or atr <= 0:
            continue
        
        tp_dist = atr * atr_tp
        sl_dist = atr * atr_sl
        
        # Walk forward
        pnl = 0.0
        exit_reason = 'timeout'
        for j in range(idx + 1, min(idx + horizon + 1, len(df))):
            if direction == 1:
                if df.loc[j, 'high'] >= entry_price + tp_dist:
                    pnl = tp_dist / entry_price
                    exit_reason = 'TP'
                    break
                if df.loc[j, 'low'] <= entry_price - sl_dist:
                    pnl = -sl_dist / entry_price
                    exit_reason = 'SL'
                    break
            else:
                if df.loc[j, 'low'] <= entry_price - tp_dist:
                    pnl = tp_dist / entry_price
                    exit_reason = 'TP'
                    break
                if df.loc[j, 'high'] >= entry_price + sl_dist:
                    pnl = -sl_dist / entry_price
                    exit_reason = 'SL'
                    break
        else:
            # Timeout: mark-to-market
            exit_price = df.loc[min(idx + horizon, len(df) - 1), 'close']
            pnl = direction * (exit_price - entry_price) / entry_price
        
        trades.append({'regime': regime, 'pnl': pnl * 100, 'direction': direction, 'exit': exit_reason})
    
    return pd.DataFrame(trades)

# Run backtests for all models
all_trades = {}
for sym in SYMBOLS:
    all_trades[sym] = {}
    for model in MODELS:
        trades_df = backtest_model_per_regime(results[sym], model)
        all_trades[sym][model] = trades_df
        print(f'{sym} {MODEL_LABELS[model]}: {len(trades_df)} trades')

print('\nAll backtests complete')

BTCUSDT MeanReversion: 107 trades
BTCUSDT Momentum: 1239 trades
BTCUSDT SqueezeBreakout: 233 trades
ETHUSDT MeanReversion: 122 trades
ETHUSDT Momentum: 1363 trades
ETHUSDT SqueezeBreakout: 251 trades

All backtests complete


In [8]:
# ── Performance matrix: model × regime group ────────────────────────
print('=' * 90)
print('MODEL PERFORMANCE BY REGIME GROUP (Proposed 4-group)')
print('=' * 90)

for sym in SYMBOLS:
    print(f'\n--- {sym} ---')
    print(f'{"Model":<18} {"Group":<14} {"Trades":>7} {"WR%":>7} {"Mean PnL%":>10} {"Sharpe":>8} {"PF":>6}')
    print('-' * 72)
    
    for model in MODELS:
        trades_df = all_trades[sym][model]
        if trades_df.empty:
            print(f'{MODEL_LABELS[model]:<18} NO TRADES')
            continue
        
        trades_df['group'] = trades_df['regime'].map(PROPOSED_GROUPS)
        
        for group in ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY']:
            g_trades = trades_df[trades_df['group'] == group]
            if len(g_trades) < 3:
                continue
            
            wr = (g_trades['pnl'] > 0).mean() * 100
            mean_pnl = g_trades['pnl'].mean()
            std_pnl = g_trades['pnl'].std()
            sharpe = mean_pnl / std_pnl if std_pnl > 0 else 0
            wins = g_trades[g_trades['pnl'] > 0]['pnl'].sum()
            losses = abs(g_trades[g_trades['pnl'] < 0]['pnl'].sum())
            pf = wins / losses if losses > 0 else float('inf')
            
            print(f'{MODEL_LABELS[model]:<18} {group:<14} {len(g_trades):>7} {wr:>6.1f}% {mean_pnl:>+9.3f}% {sharpe:>7.2f} {pf:>5.2f}')

MODEL PERFORMANCE BY REGIME GROUP (Proposed 4-group)

--- BTCUSDT ---
Model              Group           Trades     WR%  Mean PnL%   Sharpe     PF
------------------------------------------------------------------------
MeanReversion      TREND_BULL          18   11.1%    -0.788%   -1.38  0.04
MeanReversion      TREND_BEAR          23   26.1%    -0.403%   -0.46  0.37
MeanReversion      RANGE               46   34.8%    -0.253%   -0.27  0.55
MeanReversion      CHOPPY              20   50.0%    -0.027%   -0.02  0.95
Momentum           TREND_BULL         143   39.2%    -0.102%   -0.10  0.80
Momentum           TREND_BEAR         342   52.0%    +0.296%    0.19  1.59
Momentum           RANGE              476   41.2%    -0.021%   -0.02  0.95
Momentum           CHOPPY             278   42.4%    -0.034%   -0.03  0.93
SqueezeBreakout    TREND_BULL          23   43.5%    +0.198%    0.17  1.50
SqueezeBreakout    TREND_BEAR          22   40.9%    -0.134%   -0.11  0.78
SqueezeBreakout    RANGE      

In [9]:
# ── Optimal weights derivation ────────────────────────────────────
print('=' * 90)
print('OPTIMAL WEIGHT DERIVATION: Best model per regime group')
print('=' * 90)

optimal_weights = {}

for group in ['TREND_BULL', 'TREND_BEAR', 'RANGE', 'CHOPPY', 'TRANSITION']:
    best_model = None
    best_sharpe = -999
    group_results = {}
    
    for model in MODELS:
        sharpes = []
        for sym in SYMBOLS:
            trades_df = all_trades[sym][model].copy()
            if trades_df.empty:
                continue
            trades_df['group'] = trades_df['regime'].map(PROPOSED_GROUPS)
            g = trades_df[trades_df['group'] == group] if group != 'TRANSITION' else trades_df
            if len(g) >= 5:
                s = g['pnl'].mean() / g['pnl'].std() if g['pnl'].std() > 0 else 0
                sharpes.append(s)
        
        avg_sharpe = np.mean(sharpes) if sharpes else -999
        group_results[MODEL_LABELS[model]] = avg_sharpe
        if avg_sharpe > best_sharpe:
            best_sharpe = avg_sharpe
            best_model = MODEL_LABELS[model]
    
    # Derive weights: positive-Sharpe models get proportional weight, negative get 0
    positive = {k: v for k, v in group_results.items() if v > 0}
    total = sum(positive.values()) if positive else 1
    
    weights = {}
    for model_name in ['MeanReversion', 'Momentum', 'SqueezeBreakout']:
        if model_name in positive:
            weights[model_name.lower().replace('reversion', '_reversion').replace('breakout', '_breakout')] = round(positive[model_name] / total, 2)
        else:
            weights[model_name.lower().replace('reversion', '_reversion').replace('breakout', '_breakout')] = 0.0
    
    # Normalize names to match config
    w = {
        'mean_reversion': weights.get('mean_reversion', 0.0),
        'momentum': weights.get('momentum', 0.0),
        'squeeze_breakout': weights.get('squeeze_breakout', 0.0),
    }
    optimal_weights[group] = w
    
    print(f'\n{group}:')
    for k, v in group_results.items():
        marker = ' ← BEST' if k == best_model else ''
        print(f'  {k:<18}: Sharpe={v:+.3f}{marker}')
    print(f'  → Weights: {w}')

print('\n' + '=' * 90)
print('PROPOSED MODELS.YAML BLENDER CONFIG')
print('=' * 90)
print('blender:')
print('  enabled: true')
print('  transition:')
print('    entry_threshold: 0.70')
print('    exit_threshold: 0.30')
print('    floor: 0.15')
print('  mtf:')
print('    confirming_scale: 1.2')
print('    conflicting_scale: 0.5')
print('  weights:')
for group, w in optimal_weights.items():
    print(f'    {group}:')
    for k, v in w.items():
        print(f'      {k}: {v:.2f}')

OPTIMAL WEIGHT DERIVATION: Best model per regime group

TREND_BULL:
  MeanReversion     : Sharpe=-0.806
  Momentum          : Sharpe=+0.030 ← BEST
  SqueezeBreakout   : Sharpe=-0.055
  → Weights: {'mean_reversion': 0.0, 'momentum': np.float64(1.0), 'squeeze_breakout': 0.0}

TREND_BEAR:
  MeanReversion     : Sharpe=-0.635
  Momentum          : Sharpe=+0.047
  SqueezeBreakout   : Sharpe=+0.048 ← BEST
  → Weights: {'mean_reversion': 0.0, 'momentum': np.float64(0.49), 'squeeze_breakout': np.float64(0.51)}

RANGE:
  MeanReversion     : Sharpe=-0.083
  Momentum          : Sharpe=+0.046 ← BEST
  SqueezeBreakout   : Sharpe=-0.034
  → Weights: {'mean_reversion': 0.0, 'momentum': np.float64(1.0), 'squeeze_breakout': 0.0}

CHOPPY:
  MeanReversion     : Sharpe=-0.073
  Momentum          : Sharpe=+0.011
  SqueezeBreakout   : Sharpe=+0.074 ← BEST
  → Weights: {'mean_reversion': 0.0, 'momentum': np.float64(0.13), 'squeeze_breakout': np.float64(0.87)}

TRANSITION:
  MeanReversion     : Sharpe=-0.224
 

## 4. Compare Old vs New Grouping

In [10]:
# ── Simulate blended equity: old groups vs new groups ────────────
def simulate_blender(df, trades_dict, grouping, weights):
    """Simulate blended portfolio equity using regime-conditioned weights."""
    equity = [0.0]
    
    for idx in range(len(df)):
        regime = df.iloc[idx]['regime']
        group = grouping.get(regime, 'CHOPPY')
        w = weights.get(group, {})
        
        bar_pnl = 0.0
        for model, sig_col in [('mean_reversion', 'sig_mr'), ('momentum', 'sig_mom'), ('squeeze_breakout', 'sig_sq')]:
            weight = w.get(model, 0.0)
            if weight > 0 and df.iloc[idx][sig_col] != 0:
                direction = df.iloc[idx][sig_col]
                fwd = df.iloc[idx].get('fwd_ret_12', 0)
                if not np.isnan(fwd):
                    bar_pnl += weight * direction * fwd * 100
        
        equity.append(equity[-1] + bar_pnl)
    
    return equity[1:]

# Old weights (current config: momentum-only everywhere)
OLD_WEIGHTS = {g: {'mean_reversion': 0.0, 'momentum': 1.0, 'squeeze_breakout': 0.0}
               for g in ['CLEAN_TREND', 'VOLATILE_TREND', 'QUIET_RANGE', 'SQUEEZE', 'CHOPPY', 'TRANSITION']}

for sym in SYMBOLS:
    df = results[sym]
    
    eq_old = simulate_blender(df, all_trades[sym], CURRENT_GROUPS, OLD_WEIGHTS)
    eq_new = simulate_blender(df, all_trades[sym], PROPOSED_GROUPS, optimal_weights)
    
    print(f'\n{sym}:')
    print(f'  Old (momentum-only): final={eq_old[-1]:+.2f}%')
    print(f'  New (regime-cond):   final={eq_new[-1]:+.2f}%')
    print(f'  Delta:               {eq_new[-1] - eq_old[-1]:+.2f}%')


BTCUSDT:
  Old (momentum-only): final=+26.92%
  New (regime-cond):   final=+12.91%
  Delta:               -14.01%

ETHUSDT:
  Old (momentum-only): final=+162.31%
  New (regime-cond):   final=+211.50%
  Delta:               +49.19%
